# ARC-AGI-3 Agent Comparison — Option C (Full Online Sweep)

This notebook runs **ReasoningAgent** (baseline) and **WorldModelAgent** against every
game available through the ARC-AGI-3 online API and prints a side-by-side scorecard.

## Prerequisites

1. **Add Kaggle Secrets** (`Add-ons → Secrets`) with these exact names:
   - `ARC_API_KEY` — your key from [three.arcprize.org](https://three.arcprize.org/)
   - `OPENAI_API_KEY` — your OpenAI key

2. **Internet** must be ON (`Settings → Internet → On`).

3. **Accelerator** — CPU-only is fine; no GPU required.

### How it works
| Step | What happens |
|------|--------------|
| 1 | Install `arc-agi` + dependencies from `pyproject.toml` via pip |
| 2 | Clone this repo into `/kaggle/working/ARC-AGI-3-Agents` |
| 3 | Load secrets → write a `.env` file |
| 4 | Run `ReasoningAgent` (baseline) against all games, capture scorecard |
| 5 | Run `WorldModelAgent` against all games, capture scorecard |
| 6 | Display side-by-side comparison table |


## 1 — Install dependencies

In [ ]:
%%capture install_output
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "arc-agi>=0.9.1",
    "openai==1.72.0",
    "langchain[openai]>=0.3.27",
    "langgraph>=0.6.3",
    "langgraph-checkpoint-sqlite>=2.0.11",
    "numpy>=2.3.2",
    "pillow>=11.2.1",
    "pydantic>=2.11.7",
    "smolagents>=1.20.0",
    "python-dotenv",
    "requests>=2.32.4",
])
print("Dependencies installed.")

## 2 — Clone the repository

In [ ]:
import os, subprocess

REPO_DIR = "/kaggle/working/ARC-AGI-3-Agents"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call([
        "git", "clone", "--depth=1",
        "https://github.com/drQedwards/ARC-AGI-3-Agents.git",
        REPO_DIR,
    ])
    print(f"Cloned → {REPO_DIR}")
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    print(f"Updated existing clone at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 3 — Load secrets and write `.env`

Secrets must be added via **Add-ons → Secrets** in the Kaggle editor before running.

In [ ]:
from kaggle_secrets import UserSecretsClient  # available in every Kaggle kernel

secrets = UserSecretsClient()
arc_api_key   = secrets.get_secret("ARC_API_KEY")
openai_api_key = secrets.get_secret("OPENAI_API_KEY")

env_content = f"""# Auto-generated by Kaggle notebook
DEBUG=False
SCHEME=https
HOST=three.arcprize.org
PORT=443
ARC_BASE_URL=https://three.arcprize.org/
OPERATION_MODE=online
RECORDINGS_DIR=recordings
ENVIRONMENTS_DIR=environment_files
WDM_LOG=0
OPENAI_API_KEY={openai_api_key}
ARC_API_KEY={arc_api_key}
"""

with open(".env", "w") as fh:
    fh.write(env_content)

# Also export into the current process so imports that read os.environ work
os.environ["OPENAI_API_KEY"] = openai_api_key
os.environ["ARC_API_KEY"]    = arc_api_key
os.environ["OPERATION_MODE"] = "online"
os.environ["ARC_BASE_URL"]   = "https://three.arcprize.org/"

print("ARC_API_KEY   set:", bool(arc_api_key))
print("OPENAI_API_KEY set:", bool(openai_api_key))

## 4 — Verify API connection and list available games

In [ ]:
import requests

headers = {"X-API-Key": arc_api_key, "Accept": "application/json"}
r = requests.get("https://three.arcprize.org/api/games", headers=headers, timeout=15)
r.raise_for_status()

games = r.json()
game_ids = [g["game_id"] for g in games]
print(f"Connected.  Available games ({len(game_ids)}): {game_ids}")

## 5 — Helper: run an agent and capture its scorecard

We invoke `main.py` as a subprocess with `--agent`, `--tags`, and (optionally) `--game`
so that each run gets its own independent Python process with a clean environment.  
Scorecard IDs are extracted from the log output.

In [ ]:
import json, re, subprocess, sys
from pathlib import Path


def run_agent(
    agent_name: str,
    tags: str,
    game: str | None = None,
    timeout: int = 1800,
) -> dict:
    """Run *agent_name* and return {'scorecard_id', 'scorecard_url', 'log', 'summary'}."""
    cmd = [sys.executable, "main.py", f"--agent={agent_name}", f"--tags={tags}"]
    if game:
        cmd.append(f"--game={game}")

    print(f"\n>>> Starting {agent_name!r} (tags={tags!r}, game={game or 'ALL'})")

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=timeout,
        env={**os.environ, "TESTING": "False"},
    )

    full_log = result.stdout + result.stderr

    # Extract scorecard ID from log lines like:
    #   View your scorecard online: https://three.arcprize.org/scorecards/<id>
    scorecard_url = ""
    scorecard_id = ""
    for line in full_log.splitlines():
        m = re.search(r"scorecards/([\w-]+)", line)
        if m:
            scorecard_id  = m.group(1)
            scorecard_url = f"https://three.arcprize.org/scorecards/{scorecard_id}"

    # Extract final SCORECARD REPORT JSON block
    summary: dict = {}
    json_match = re.search(
        r"FINAL SCORECARD REPORT.*?({.*?})",
        full_log, re.DOTALL
    )
    if json_match:
        try:
            summary = json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass

    print(f"    Finished.  Return code: {result.returncode}")
    if scorecard_url:
        print(f"    Scorecard: {scorecard_url}")

    return {
        "agent": agent_name,
        "tags": tags,
        "scorecard_id": scorecard_id,
        "scorecard_url": scorecard_url,
        "return_code": result.returncode,
        "log": full_log,
        "summary": summary,
    }


print("Helper defined.")

## 6 — Run ReasoningAgent (baseline) — all games

In [ ]:
result_baseline = run_agent(
    agent_name="reasoningagent",
    tags="baseline,optionC,full",
)

## 7 — Run WorldModelAgent — all games

In [ ]:
result_worldmodel = run_agent(
    agent_name="worldmodelagent",
    tags="worldmodel,optionC,full",
)

## 8 — Side-by-side comparison

In [ ]:
import pandas as pd


def extract_metrics(run: dict) -> dict:
    """Pull key metrics from a run result dict."""
    s = run.get("summary", {})
    # EnvironmentScorecard fields (best-effort; field names may vary by version)
    return {
        "agent": run["agent"],
        "tags": run["tags"],
        "scorecard_id": run["scorecard_id"] or "(not captured)",
        "scorecard_url": run["scorecard_url"] or "(not captured)",
        "levels_completed": s.get("levels_completed", s.get("score", "—")),
        "win_levels": s.get("win_levels", s.get("win_score", "—")),
        "total_actions": s.get("total_actions", "—"),
        "games_played": len(s.get("game_results", [])),
    }


rows = [extract_metrics(result_baseline), extract_metrics(result_worldmodel)]
df = pd.DataFrame(rows).set_index("agent")

print("\n===== COMPARISON RESULTS =====\n")
print(df.to_string())

print("\n===== SCORECARD LINKS =====\n")
for r in [result_baseline, result_worldmodel]:
    print(f"  {r['agent']:20s}  {r['scorecard_url'] or '(not captured)'}")

## 9 — Save results to disk

In [ ]:
import json
from datetime import datetime

timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
output_path = Path("/kaggle/working") / f"comparison_results_{timestamp}.json"

# Strip large log blobs before saving to keep the file readable
saveable = []
for r in [result_baseline, result_worldmodel]:
    entry = {k: v for k, v in r.items() if k != "log"}
    saveable.append(entry)

output_path.write_text(json.dumps(saveable, indent=2))
print(f"Results saved → {output_path}")

# Also save per-agent logs
for r in [result_baseline, result_worldmodel]:
    log_path = Path("/kaggle/working") / f"{r['agent']}_{timestamp}.log"
    log_path.write_text(r["log"])
    print(f"Log saved → {log_path}")

## 10 — (Optional) Single-game comparison

Run the two agents on a single game for a faster, focused comparison.  
Change `GAME` to any `game_id` from the list printed in Step 4.

In [ ]:
GAME = "locksmith"   # ← change to any game_id from Step 4

result_single_baseline = run_agent(
    agent_name="reasoningagent",
    tags=f"baseline,optionC,single,{GAME}",
    game=GAME,
)

result_single_worldmodel = run_agent(
    agent_name="worldmodelagent",
    tags=f"worldmodel,optionC,single,{GAME}",
    game=GAME,
)

rows_single = [
    extract_metrics(result_single_baseline),
    extract_metrics(result_single_worldmodel),
]
df_single = pd.DataFrame(rows_single).set_index("agent")

print(f"\n===== SINGLE-GAME COMPARISON ({GAME}) =====\n")
print(df_single.to_string())